Input df should looks like:

[[feature1, feature2, feature 3, ..., personID, session, class]
[]
[]
[...]]

In [88]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix, accuracy_score, auc, roc_curve

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
# from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [90]:
import models_train as tr

In [91]:
pd.set_option('display.max.columns',100)
pd.set_option('float_format', '{:.2f}'.format)

# Change input paramethers (data folder, column names)

In [75]:
df = pd.read_csv('ecg_features_final_standard.csv')
df.columns

Index(['participant', 'session', 'label', 'ecg_mean', 'ecg_std', 'ecg_min',
       'ecg_max', 'ecg_range', 'ecg_median', 'ecg_q25', 'ecg_q75', 'ecg_iqr',
       'ecg_rms', 'ecg_energy', 'ecg_mad', 'ecg_abs_diff_mean',
       'ecg_abs_diff_std', 'ecg_line_length', 'ecg_peak_count',
       'ecg_peak_rate', 'ecg_peak_mean', 'ecg_peak_std'],
      dtype='object')

In [92]:
df.head()

,participant,session,label,ecg_mean,ecg_std,ecg_min,ecg_max,ecg_range,ecg_median,ecg_q25,ecg_q75,ecg_iqr,ecg_rms,ecg_energy,ecg_mad,ecg_abs_diff_mean,ecg_abs_diff_std,ecg_line_length,ecg_peak_count,ecg_peak_rate,ecg_peak_mean,ecg_peak_std
0,olha,session3_arm_GOOD,0,0.04,0.95,-2.31,8.22,10.53,-0.18,-0.35,0.05,0.40,0.95,6993.18,0.50,0.36,0.83,2823.70,419,7.04,2.23,1.89
1,olha,session3_arm_GOOD,0,0.02,0.95,-2.31,8.22,10.53,-0.19,-0.35,0.02,0.37,0.95,7001.70,0.50,0.37,0.84,2862.69,427,7.19,2.23,1.89
2,olha,session3_arm_GOOD,0,0.02,0.95,-2.31,8.22,10.53,-0.19,-0.35,0.02,0.37,0.95,7008.30,0.51,0.36,0.82,2813.29,414,6.96,2.24,1.90
3,olha,session3_arm_GOOD,0,0.01,0.95,-2.31,8.22,10.53,-0.21,-0.36,0.00,0.37,0.95,6987.67,0.51,0.36,0.82,2776.33,403,6.77,2.27,1.91
4,olha,session3_arm_GOOD,0,0.02,0.95,-2.31,8.22,10.53,-0.19,-0.36,0.00,0.37,0.95,6989.52,0.51,0.36,0.82,2782.06,411,6.90,2.25,1.90


In [76]:
ID_col = "participant"
session_col = "session"
target_col = 'label'

drop_cols = [ID_col, session_col]

In [77]:
df.label.value_counts()

label
0    641
3    300
1    203
2    122
Name: count, dtype: int64

In [78]:
#label column to binary
df['label'] = df['label'].apply(lambda x: 0 if x == 0 else 1)
df.label.value_counts()


label
0    641
1    625
Name: count, dtype: int64

# Specify which model do you want to use and add their paramethers grid

In [79]:
model_configs = {
    "LogReg": {
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "params": {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear']
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 5, 10],
            'min_samples_split': [2, 5, 10]
        }
    },
    "DecisionTree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            'max_depth': [None, 5, 10, 15, 20],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'criterion': ['gini', 'entropy']
        }
    },
    "XGBoost": {
        "model": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
        "params": {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.1, 0.2]
        }
    },
    # "SVC": {
    #     "model": SVC(probability=True, class_weight="balanced", random_state=42),
    #     "params": {
    #         'C': [0.1, 1, 10],
    #         'kernel': ['rbf', 'linear'],
    #         'gamma': ['scale', 'auto']
    #     }
    # }
}

# JUST RUN (NO changes required)

In [80]:
logo = LeaveOneGroupOut()

X = df.drop(columns=[target_col])
y = df[target_col]
groups = df[ID_col]
print(f"X shape: {X.shape}, y shape: {y.shape}")


X shape: (1266, 21), y shape: (1266,)


In [ ]:
# Run LOSO
scores = tr.train_loso(
    model_configs=model_configs,
    X=X,
    y=y,
    groups=groups,
    logo=logo,
    drop_cols=drop_cols # MAYBE YOU NEED TO ADD MORE COLUMNS HERE 
)

LogReg: {'solver': 'liblinear', 'penalty': 'l2', 'C': 1}
RandomForest: {'n_estimators': 50, 'min_samples_split': 5, 'max_depth': 10}
DecisionTree: {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 5, 'criterion': 'gini'}
XGBoost: {'n_estimators': 50, 'max_depth': 5, 'learning_rate': 0.01}
LogReg: {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.01}
RandomForest: {'n_estimators': 50, 'min_samples_split': 2, 'max_depth': 10}
DecisionTree: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 20, 'criterion': 'entropy'}
XGBoost: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.01}
LogReg: {'solver': 'liblinear', 'penalty': 'l2', 'C': 10}
RandomForest: {'n_estimators': 50, 'min_samples_split': 2, 'max_depth': None}
DecisionTree: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'gini'}
XGBoost: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.01}


In [ ]:
#PRINT SUMMARY
df_train, df_val = tr.summarize_scores(scores)


--- Training Metrics ---

Model: DecisionTree
f1        : 0.8905 ± 0.1897
recall    : 0.8633 ± 0.2368
precision : 0.9264 ± 0.1274
accuracy  : 0.9071 ± 0.1609
roc_auc   : 0.9316 ± 0.1184

Model: LogReg
f1        : 0.5775 ± 0.0672
recall    : 0.5874 ± 0.1002
precision : 0.5708 ± 0.0412
accuracy  : 0.5787 ± 0.0439
roc_auc   : 0.6110 ± 0.0398

Model: RandomForest
f1        : 0.9920 ± 0.0076
recall    : 0.9924 ± 0.0076
precision : 0.9916 ± 0.0077
accuracy  : 0.9920 ± 0.0074
roc_auc   : 0.9998 ± 0.0003

Model: XGBoost
f1        : 0.8726 ± 0.0789
recall    : 0.8507 ± 0.1135
precision : 0.8987 ± 0.0424
accuracy  : 0.8815 ± 0.0656
roc_auc   : 0.9514 ± 0.0379

--- Validation Metrics ---

Model: DecisionTree
f1        : 0.6136 ± 0.0560
recall    : 0.6644 ± 0.1804
precision : 0.5888 ± 0.0381
accuracy  : 0.5927 ± 0.0288
roc_auc   : 0.5808 ± 0.0420

Model: LogReg
f1        : 0.2837 ± 0.3047
recall    : 0.3437 ± 0.4656
precision : 0.3724 ± 0.1783
accuracy  : 0.4766 ± 0.0258
roc_auc   : 0.4779 ± 0.06

In [86]:
df_train.sort_values(by=['model'])

,f1,recall,precision,accuracy,roc_auc,model
2,0.67,0.59,0.78,0.72,0.79,DecisionTree
6,1.00,1.00,1.00,1.00,1.00,DecisionTree
10,1.00,1.00,1.00,1.00,1.00,DecisionTree
0,0.50,0.48,0.53,0.54,0.58,LogReg
4,0.62,0.67,0.58,0.57,0.60,LogReg
8,0.61,0.62,0.61,0.63,0.66,LogReg
1,0.98,0.98,0.98,0.99,1.00,RandomForest
5,0.99,0.99,0.99,0.99,1.00,RandomForest
9,1.00,1.00,1.00,1.00,1.00,RandomForest
3,0.79,0.73,0.87,0.81,0.91,XGBoost


In [87]:
df_val.sort_values(by=['model'])

,f1,recall,precision,accuracy,roc_auc,model
2,0.68,0.87,0.56,0.58,0.54,DecisionTree
6,0.59,0.59,0.58,0.63,0.62,DecisionTree
10,0.58,0.53,0.63,0.58,0.58,DecisionTree
0,0.01,0.00,0.17,0.48,0.51,LogReg
4,0.61,0.87,0.47,0.50,0.52,LogReg
8,0.23,0.15,0.48,0.45,0.41,LogReg
1,0.55,0.50,0.62,0.59,0.62,RandomForest
5,0.60,0.74,0.51,0.57,0.58,RandomForest
9,0.65,0.71,0.60,0.58,0.57,RandomForest
3,0.44,0.37,0.55,0.52,0.62,XGBoost
